Import packages, specify directories

In [ ]:
# For data importing and feature extraction
import librosa
import os 
import numpy as np 
import pandas as pd 
import joblib

# For model development and training 
import xgboost as xgb
from keras.models import Sequential, Model
from keras.layers import Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
import tensorflow as tf
import optuna

# For PCA analysis
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt 
from sklearn.metrics import silhouette_score, accuracy_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import GroupShuffleSplit

# Set the data download directory for reading 
datapath = 'Data/genres_original/'

# Define a fixed sample rate and duration for target training
sr = 22_050
duration = 30 
hop = 2_048

Define an Audiofile class containing preliminary feature extractions

In [ ]:
class Audiofile:
    """
    A class containing an audio file and feature attributes

    ...

    Methods
    ----------

    extract_stat_features() -> tuple[np.array]
        Computes statistical embeddings of audio file features:
            Mel-Frequency Cepstral Coefficients (MFCCs)
            Mel-Frequency Cepstral Coefficient Deltas (MFCC Deltas)
            Tempo 
            Frequency Rolloff at 1% (Minimum)
            Frequency Rolloff at 99% (Maximum)
            Root Mean Square
            Zero-Crossing Rate
            Tone Network (Tonnetz)
            Spectral Contrast 
            Centre / Centroid Frequency

    @staticmethod
    
    fix_length(y: np.array, target: int) -> np.array
        Trims or 0-pads an array.

    fix_frames(spec: np.array) -> np.array:
        Trims or 0-pads an image / 2-dimensional array).

    normalize_spectogram(spec: np.array) -> np.array
        Normalizes and scales an image / 2-dimensional array by transforming to the minimum value and mapping to [0, 1].

    extract_spec_features() -> tuple[np.array]:
        Computes the spectral renderings of audio file features:
            Mel-Frequency Spectogram, normalized to dB log-10 scale
            Tempogram 
            Chromagram 
    """
    def __init__(self, series: np.array, sr: int, duration: int, spec_hop: int) -> None:
        self.y = series 
        self.sr = sr 
        self.duration = duration
        self.spec_hop = spec_hop

    def __str__(self):
        return 'Audiofile object'


    def extract_stat_features(self) -> tuple[np.array]:
        """
        Computes the Librosa features of an audio file.
        """
        mfcc = librosa.feature.mfcc(y = self.y, sr = self.sr, n_mfcc = 20)
        mfcc_deltas = librosa.feature.delta(mfcc)
        tempo = librosa.feature.tempo(y = self.y, sr = self.sr)
        rolloff_001 = librosa.feature.spectral_rolloff(y = self.y, sr = self.sr, roll_percent = 0.01)
        rolloff_099 = librosa.feature.spectral_rolloff(y = self.y, sr = self.sr, roll_percent = 0.99)
        rms = librosa.feature.rms(y = self.y)
        zcr = librosa.feature.zero_crossing_rate(y = self.y)
        tonnetz = librosa.feature.tonnetz(y = self.y, sr = self.sr)
        contrast_bands = librosa.feature.spectral_contrast(y = self.y, sr = self.sr)
        centroid_freq = librosa.feature.spectral_centroid(y = self.y, sr = self.sr)

        return mfcc, mfcc_deltas, tempo, rolloff_001, rolloff_099, rms, zcr, tonnetz, contrast_bands, centroid_freq


    # Calling outside the class 
    @staticmethod
    def fix_length(y: np.array, target: int = sr * duration) -> np.array:
        """
        Fix the length of an array.
        """
        if len(y) > target:
            return y[ : target]
        return np.pad(y, (0, target - len(y)))

    def fix_frames(self, spec: np.array) -> np.array:
        """
        Fix the length and width of an image / 2-dimensional array.
        """
        target = 1 + (self.sr * self.duration) // self.spec_hop

        time_dim = spec.shape[1]
        if time_dim > target:
            return spec[:, :target]
        return np.pad(spec, ((0,0), (0, target - time_dim)))
    

    def normalize_spectogram(self, spec: np.array) -> np.array:
        mn, mx = spec.min(), spec.max()
        if mx - mn < 1e-8:
            return np.zeros_like(spec)
        return (spec - mn) / (mx - mn)


    def extract_spec_features(self) -> tuple[np.array]:
        tempogram = librosa.feature.tempogram(y = self.y, sr = self.sr)
        mel_spec = librosa.feature.melspectrogram(y = self.y, sr = self.sr)
        mel_spec = librosa.power_to_db(mel_spec, ref = np.max)
        chromagram = librosa.feature.chroma_cens(y = self.y, sr = self.sr)
        
        return (self.normalize_spectogram(self.fix_frames(tempogram)), 
                self.normalize_spectogram(self.fix_frames(mel_spec)), 
                self.normalize_spectogram(self.fix_frames(chromagram)))

Build an array of Audiofile objects with class labels. Perform segment augmentation and a train/test split.

In [ ]:
# Define an audio dictionary for storing Audiofile objects
audio_dict = {
    'label' : [],
    'track_id' : [],
    'audio_object' : []
}

# Iterate through the data directory and load every file 
for dirpath, dirnames, filenames in os.walk(datapath):
    label = os.path.basename(dirpath)
    for file in filenames:
        try:
            # Fixed sample rate implies we only need to pull y, the time series
            y, _ = librosa.load(datapath + f'{label}/{file}', duration = duration, sr = sr)
            # Fix 3 second intervals for segment augmentation
            seg_len = sr * 3
            for i, start in enumerate(range(0, len(y)-seg_len+1, seg_len)):
                # Slice the interval and append
                seg = y[start:start+seg_len]
                audio_dict['label'].append(label)
                audio_dict['track_id'].append(f'{label}/{file}')   # for group split
                audio_dict['audio_object'].append(Audiofile(series=seg, sr=sr, duration=3, spec_hop=hop))
        except Exception as e:
            print(f'Error opening file {file}: {e}.')

# Convert the dictionary into a Pandas dataframe for easy data manipulation
AUDIO = pd.DataFrame(audio_dict)
display(AUDIO.head(15))

# Write a label encoder to map the music genres to integers
le = LabelEncoder()
y_enc = le.fit_transform(AUDIO['label'].values)

# Since we are using segmented augmentation on the clips, we need to group each audiofile with the respected track id
# Or else our train/test split will contain a random mix of different 3s samples from various genres instead of the same genre
gss = GroupShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 42)
# Pull indices 
train_idx, test_idx = next(gss.split(AUDIO, y_enc, groups = AUDIO['track_id']))


Implement CNNs on image data

In [ ]:
# Stack data from image to fixed length 
tempo_arr_pre = np.stack([a.extract_spec_features()[0] for a in AUDIO['audio_object'].values])
mel_arr_pre = np.stack([a.extract_spec_features()[1] for a in AUDIO['audio_object'].values])
chroma_arr_pre = np.stack([a.extract_spec_features()[2] for a in AUDIO['audio_object'].values])

In [ ]:
# Add a new axis on the image to begin resizing and fit to the 2-dimensional convolution on the Sequential()
tempo_arr = tempo_arr_pre[..., np.newaxis]
mel_arr = mel_arr_pre[..., np.newaxis]
chroma_arr = chroma_arr_pre[..., np.newaxis]

# Use Tensorflow to resize the array
tempo_small = tf.image.resize(tempo_arr, [128, 128]).numpy()
mel_small   = tf.image.resize(mel_arr,   [128, 128]).numpy()
chroma_small= tf.image.resize(chroma_arr,[12, 128]).numpy() # 12 due to octave range in Western music

# Convert datatypes to preserve memory
tempo_small = tempo_small.astype('float32')
mel_small   = mel_small.astype('float32')
chroma_small= chroma_small.astype('float32')

# Extract input layer dimensions
n_temp = tempo_small.shape[1]   # 384  (tempogram lag bins)
n_mel  = mel_small.shape[1]     # 128  (mel bands)
n_chroma = chroma_small.shape[1]# 12   (pitch classes)
T = tempo_small.shape[2]        # time frames (shared)

In [ ]:
# 1. Tempogram CNN
model_tempogram = Sequential([
    Input(shape=(n_temp, T, 1)),

    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    GlobalAveragePooling2D(),
    
    Dense(64, activation='relu', name='embedding'), # We will use this layer to pull embeddings from; name it
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model_tempogram.summary()

In [ ]:
# 2. Mel Spectogram CNN
# This spectogram is the most important feature - make the CNN deep!
model_mel = Sequential([
    Input(shape=(n_mel, T, 1)),

    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    GlobalAveragePooling2D(),

    Dense(128, activation='relu', name='embedding'), # We will use this layer to pull embeddings from; name it
    Dropout(0.4),
    Dense(10, activation='softmax')
])

model_mel.summary()

In [ ]:
# 3. CENS Chromagram CNN
model_chroma = Sequential([
    Input(shape=(n_chroma, T, 1)),

    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    GlobalAveragePooling2D(),
    
    Dense(64, activation='relu', name='embedding'), # We will use this layer to pull embeddings from; name it
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model_chroma.summary()

In [ ]:
# Compile the models
model_tempogram.compile(loss='sparse_categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])
model_mel.compile(loss='sparse_categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])
model_chroma.compile(loss='sparse_categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [ ]:
# Train
model_tempogram.fit(x = tempo_small[train_idx], y = y_enc[train_idx], batch_size = 16, epochs = 10, verbose = 1)


In [ ]:
# Train
model_mel.fit(x = mel_small[train_idx], y = y_enc[train_idx], batch_size = 16, epochs = 10, verbose = 1)

In [ ]:
# Train 
model_chroma.fit(x = chroma_small[train_idx], y = y_enc[train_idx], batch_size = 16, epochs = 10, verbose = 1)

Extract model from CNN

In [ ]:
# Define an extraction function which pulls the output layer into an embedded vector
def make_extractor(model: Sequential) -> Model:
    return Model(inputs = model.inputs, outputs = model.get_layer('embedding').output)

# Pull the embedding layers
tempo_ext = make_extractor(model_tempogram)
mel_ext    = make_extractor(model_mel)
chroma_ext = make_extractor(model_chroma)

# Get the embeddings from training
tempo_emb  = tempo_ext.predict(tempo_small,  batch_size=16)  
mel_emb    = mel_ext.predict(mel_small,       batch_size=16)   
chroma_emb = chroma_ext.predict(chroma_small, batch_size=16)  

# Concatenate all the embedding vectors into the same np.array
X_all = np.concatenate([tempo_emb, mel_emb, chroma_emb], axis=1) 

# Perform the train/test split using previously computed indices
X_train_cnn, X_test_cnn = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_enc[train_idx], y_enc[test_idx]


Extract statistical features from audio objects

In [ ]:
# Define an aggregation function to compute statistical values of an array
def aggregate(array: np.array) -> np.array:
    """
    Compute statistical aggregations from an array
    ...

    Mean
    Standard Deviation
    Maximum
    Minimum
    """
    return np.concatenate([array.mean(axis = 1), array.std(axis = 1), array.max(axis = 1), array.min(axis = 1)])


# Initialize a blank list to store the statistical vectors in
stat_vectors = []

# Iterate over all audio clips
for audio in AUDIO['audio_object'].values:
    # Get feature arrays from the tuple
    mfcc, mfcc_deltas, tempo, rolloff_001, rolloff_099, rms, zcr, tonnetz, contrast_bands, centroid_freq = audio.extract_stat_features()

    # Arrange an entire vector to store all aggregations in (4x number of features!)
    features = np.concatenate([
        aggregate(mfcc),
        aggregate(mfcc_deltas),
        aggregate(rolloff_001),
        aggregate(rolloff_099),
        aggregate(rms),
        aggregate(zcr),
        aggregate(tonnetz),
        aggregate(contrast_bands),
        aggregate(centroid_freq),
        np.atleast_1d(tempo)
    ])

    # Append to all statistical vectors
    stat_vectors.append(features)

# Convert list to np.array
X_stat = np.array(stat_vectors)
# Verify shape
print(X_stat.shape)

Fuse CNN embeddings and statistical vectors together into one feature pool

In [ ]:
# Scale the statistics
scaler = StandardScaler()
# Scale the input to set the scaler, then transform the testing set
X_stat_train = scaler.fit_transform(X_stat[train_idx])
X_stat_test = scaler.transform(X_stat[test_idx])

# Combine statistical features and neural network features
X_train = np.concatenate([X_train_cnn, X_stat_train], axis = 1)
X_test = np.concatenate([X_test_cnn, X_stat_test], axis = 1)

Implement XGBoost: determine the feature scores, train the model, prevent overfitting, conduct hyperparameter tuning, K-folds

In [ ]:
# # SKIP
# model = xgb.XGBClassifier()

# param_dist = {
#     'max_depth' : [3, 6, 10, 15],
#     'learning_rate' : [0.5, 0.1, 0.01, 0.001],
#     'subsample' : [0.25, 5, 0.75, 1]
# }

# grid_search = GridSearchCV(estimator = model, param_grid = param_dist, cv = 5, scoring = 'accuracy')

# grid_search.fit(X_train, y_train)
# print('Best parameters:', grid_search.best_params_)
# print('Best score:', grid_search.best_score_)

# # Only yields a maximum accuracy of 0.78 - try to tune the model a different way.

In [ ]:
# Try tuning with Optuna
def objective(trial) -> float:
    """
    A function to perform tuning on XGBoost with hyperparameter grid search
    """
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    # Define the XGBClassifier instance
    m = xgb.XGBClassifier(**params, eval_metric='mlogloss')
    # Fit the function
    m.fit(X_train, y_train)
    return accuracy_score(y_test, m.predict(X_test))

# Optimize the model & return the best parameters
study = optuna.create_study(direction = 'maximize', study_name = 'music_genre_xgb_tuning')
study.optimize(objective, n_trials = 5)
print(study.best_params)

In [ ]:
# Initialize an XGBoost instance with optimized hyperparameters
model = xgb.XGBClassifier(n_estimators = 696, max_depth = 4, learning_rate = 0.06225907507238952, subsample = 0.6244548917675014, colsample_bytree = 0.6730345350529875)

# Fit the model
model.fit(X_train, y_train)

# Input test dataset and return predictions
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

Perform linear discriminant analysis across classes 

In [ ]:
# Input all data (train + test) with label encodings
X = np.concatenate([X_all, X_stat], axis = 1)
y = np.array(y_enc)

# Transform all of the data
X_scaled = StandardScaler().fit_transform(X, y_enc)

# Initialize a linear discriminant analysis object
lda = LDA(n_components = 2)
# Scale the LDA
X_lda = lda.fit_transform(X_scaled, y)

# Plot the class overlap and seperability for 2 features
for i, g in enumerate(le.classes_):
    m = y == i
    plt.scatter(X_lda[m,0], X_lda[m,1], s=8, label=g)
plt.legend(loc='lower right', fontsize = 8)
plt.xlabel('LD1')
plt.ylabel('LD2')
plt.title('Linear Discriminant Analysis Plot for Music Genres', fontsize = 10)
plt.show()

# Now, perform an LDA on all 9 features instead of just 2
lda_full = LDA(n_components = 9)
X_lda_full = lda_full.fit_transform(X_scaled, y)

# Compute Silhouette score to measure the class separation
score = silhouette_score(X_lda_full, y)
print('Silhouette score:', score)

Export and save all models, neural networks, and parameters

In [ ]:
# Export the models with .save and joblib
model_mel.save('music-genre-classifier/artifacts/model_mel.keras')
model_tempogram.save('music-genre-classifier/artifacts/model_tempogram.keras')
model_chroma.save('music-genre-classifier/artifacts/model_chroma.keras')

joblib.dump(model, 'music-genre-classifier/artifacts/xgbmodel.pkl')
joblib.dump(scaler, 'music-genre-classifier/artifacts/scaler.pkl')
joblib.dump(le, 'music-genre-classifier/artifacts/label_encoder.pkl')